## 2.4 Polar 编码性能对比与分析

在上一节中，我们建立了无编码 BPSK 的 BER 基线。本节引入 Polar 编码，先走通一次完整的编解码链路，再进行 SNR 扫描定量标定编码增益，最后遍历不同码率与码长分析编码参数对性能的影响。

本节学习大纲如下：

- Polar 编解码一次演示
- Polar vs 无编码 BER 扫描
- 编码增益分析
- 多码率与多码长对比

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── common/
│   └── polar.py             <- PolarEncoder: Arikan 蝶形编码
│                               PolarDecoder: SC 逐次消除解码 (f/g 递推)
├── phy/
│   ├── psk.py               <- PSKModulator / PSKDemodulator: BPSK 调制解调
│   └── channel.py           <- ChannelModel: AWGN 信道模型
```


---

### 1. 导入与参数

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.common.polar import PolarEncoder, get_polar_decoder, get_info_bit_count
from nearlink_sdr.phy.psk import PSKModulator, PSKDemodulator
from nearlink_sdr.phy.channel import ChannelModel

---

### 2. Polar 编解码流程演示

以 Polar(256, 112) R=1/2 为例，完成 信息比特 -> 编码 -> BPSK -> AWGN -> LLR -> SC 解码 的完整链路。

In [ ]:
N, K = 256, 112
enc = PolarEncoder(N, K)
dec = get_polar_decoder(N, K)
rng = np.random.default_rng(42)

# Polar 编码
info = rng.integers(0, 2, size=K, dtype=np.int8)
coded = enc.encode(info)
print(f"K={K} -> N={N}, 码率 R={K/N:.3f}")

# BPSK 调制 + AWGN + LLR
bpsk = 1.0 - 2.0 * coded.astype(np.float64)
snr_db = 2.0
R_val = K / N
snr_lin = 10.0 ** (snr_db / 10.0)
noise_var = 1.0 / (2.0 * R_val * snr_lin)
noise = rng.normal(0, np.sqrt(noise_var), size=N)
received = bpsk + noise
llr = 2.0 * received / noise_var

# SC 解码
decoded = dec.decode(llr)
errors = int(np.sum(decoded != info))
print(f"LLR (前8): {np.round(llr[:8], 2)}")
print(f"原始信息 (前20): {info[:20]}")
print(f"SC 解码   (前20): {decoded[:20]}")
print(f"错误比特: {errors} / {K}")

### 关键函数说明

上述演示中涉及的三个核心环节：

---

**Polar 编码：`enc.encode(info)`**

将 K 个信息比特排列到预设的好信道位置上（冻结位填 0），通过蝶形 GF(2) 变换生成 N 位码字。

```python
def encode(self, info_bits: np.ndarray) -> np.ndarray:
    u = np.zeros(self.N, dtype=np.int8)
    u[self._info_pos_arr] = info_bits             # 向量化插入, O(1)
    d = u.copy()
    stage = 1
    while stage < self.N:                           # 逐级异或, 共 log2(N) 级
        for j in range(0, self.N, 2 * stage):
            d[j:j + stage] ^= d[j + stage:j + 2 * stage]
        stage <<= 1
    return d
```

其中 `self._info_pos_arr` 存储 K 个好信道的索引，由可靠性序列决定；其余 N-K 个冻结位始终为 0。

---

**SNR 归一化（含码率补偿）**

这是 Polar 仿真中最容易出错的一步。由于每信息比特分配到的能量为 Eb/R（R 为码率），噪声方差必须按码率缩放：

noise_var = 1 / (2 * R * SNR_lin)

```python
R_val = K / N                                       # 码率
snr_lin = 10.0 ** (snr_db / 10.0)                   # dB -> 线性
noise_var = 1.0 / (2.0 * R_val * snr_lin)           # 码率补偿的噪声方差
noise = rng.normal(0, np.sqrt(noise_var), size=N)    # 实高斯噪声
received = bpsk + noise
llr = 2.0 * received / noise_var                     # BPSK LLR: 2*y/sigma^2
```

不加码率补偿时，低码率会高估每符号 SNR，导致仿真 BER 偏低。`noise_var` 中的 R 因子是将总能量按信息比特归一化的关键。

---

**SC 解码：`dec.decode(llr)`**

接收 N 个信道 LLR，通过蝶形 f/g 递推逐比特判决。

```python
class PolarDecoder:
    def __init__(self, N: int, K: int) -> None:
        self.N = N; self.K = K
        self.n = int(np.log2(N))              # 蝶形层数
        self._is_frozen = np.ones(N, dtype=bool)
        for pos in info_positions:
            self._is_frozen[pos] = False      # 标记信息位
        self._L = np.zeros((self.n + 1, N), dtype=np.float64)   # LLR 数组
        self._B = np.zeros((self.n + 1, N), dtype=np.int8)      # 部分和数组
```

`decode(llr)` 将信道 LLR 填入 `_L[n]` 层，逐层向左下行计算 f/g 操作，最终在 `_L[0]` 层逐比特判决：冻结位判 0，信息位根据 LLR 符号判 0/1。工厂函数 `get_polar_decoder(N, K)` 等价于 `PolarDecoder(N, K)`。

```python
dec = get_polar_decoder(256, 112)
decoded = dec.decode(llr)
```

---

### 3. Polar vs 无编码 BER 扫描

In [ ]:
snr_range = np.arange(-4, 12, 1)
enc = PolarEncoder(256, 112)
dec = get_polar_decoder(256, 112)
n_blocks = max(1, 5000 // 112)
rng = np.random.default_rng(42)

ber_coded, fer_coded = [], []
for snr in snr_range:
    total_err, total_bits, frame_err = 0, 0, 0
    for _ in range(n_blocks):
        info = rng.integers(0, 2, size=112, dtype=np.int8)
        coded = enc.encode(info)
        tx = 1.0 - 2.0 * coded.astype(np.float64)
        R_v = 112 / 256
        snr_lin = 10.0 ** (float(snr) / 10.0)
        nv = 1.0 / (2.0 * R_v * snr_lin)
        noise = rng.normal(0, np.sqrt(nv), size=256)
        rx = tx + noise
        llr = 2.0 * rx / nv
        decoded = dec.decode(llr)
        errs = int(np.sum(decoded != info))
        total_err += errs; total_bits += 112
        if errs > 0: frame_err += 1
    ber_coded.append(total_err / total_bits if total_bits > 0 else 0.0)
    fer_coded.append(frame_err / n_blocks)

mod = PSKModulator(mod_type="BPSK", sps=4)
demod = PSKDemodulator(mod_type="BPSK", sps=4)
rng2 = np.random.default_rng(42)
tx_bits = rng2.integers(0, 2, 5000)
ber_uncoded = []
for snr in snr_range:
    tx_sig = mod.modulate(tx_bits)
    ch = ChannelModel(snr_db=float(snr))
    rx_sig = ch.apply_awgn(tx_sig, 4)
    rx_b = demod.demodulate(rx_sig)
    n = min(len(tx_bits), len(rx_b))
    ber_uncoded.append(np.mean(tx_bits[:n] != rx_b[:n]))

print(f"{'SNR':>5s}  {'Uncoded':>10s}  {'Polar':>10s}  {'FER':>8s}")
for s, bu, bp, fp in zip(snr_range, ber_uncoded, ber_coded, fer_coded):
    print(f"{s:5.0f}  {bu:10.6f}  {bp:10.6f}  {fp:8.4f}")

---

### 4. 对比曲线与编码增益

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(snr_range, [max(b, 1e-6) for b in ber_uncoded], "o-", label="Uncoded BPSK")
ax1.semilogy(snr_range, [max(b, 1e-6) for b in ber_coded], "s-", label="Polar R=1/2")
ax1.set_xlabel("Eb/N0 (dB)"); ax1.set_ylabel("Bit Error Rate")
ax1.set_title("Polar Coded vs Uncoded BPSK BER")
ax1.legend(); ax1.grid(True, which="both", ls="--", alpha=0.5); ax1.set_ylim(bottom=1e-5)

ax2.semilogy(snr_range, [max(f, 1e-4) for f in fer_coded], "s-", label="Polar R=1/2 FER")
ax2.set_xlabel("Eb/N0 (dB)"); ax2.set_ylabel("Frame Error Rate")
ax2.set_title("Polar Coded FER")
ax2.legend(); ax2.grid(True, which="both", ls="--", alpha=0.5); ax2.set_ylim(bottom=1e-4)
plt.tight_layout(); plt.show()


---

### 5. 多配置对比

遍历不同码率与码长的组合，观察编码参数对 BER 的影响。

In [ ]:
snr_range2 = np.arange(-2, 10, 0.5)
configs = [
    ("1/2", 256, "Polar(256,112) R=1/2"),
    ("1/2", 512, "Polar(512,224) R=1/2"),
    ("3/4", 256, "Polar(256,176) R=3/4"),
    ("1/4", 256, "Polar(256,48)  R=1/4"),
]

results = []
for rate, length, label in configs:
    Kv = get_info_bit_count(rate, length)
    e = PolarEncoder(length, Kv)
    d = get_polar_decoder(length, Kv)
    rg = np.random.default_rng(42)
    nb = max(1, 5000 // Kv)
    ber_list = []
    for snr in snr_range2:
        terr, tbit = 0, 0
        for _ in range(nb):
            info = rg.integers(0, 2, size=Kv, dtype=np.int8)
            coded = e.encode(info)
            tx = 1.0 - 2.0 * coded.astype(np.float64)
            R_v = Kv / length
            snr_lin = 10.0 ** (float(snr) / 10.0)
            nv = 1.0 / (2.0 * R_v * snr_lin)
            noise = rg.normal(0, np.sqrt(nv), size=length)
            rx = tx + noise
            llr = 2.0 * rx / nv
            decoded = d.decode(llr)
            terr += int(np.sum(decoded != info))
            tbit += Kv
        ber_list.append(terr / tbit if tbit > 0 else 0.0)
    results.append((label, ber_list))
    print(f"{label}: done")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
markers = ["o-", "s-", "^-", "d-"]
for (label, ber), mk in zip(results, markers):
    ax.semilogy(snr_range2, [max(b, 1e-6) for b in ber], mk, label=label, markersize=3)
ax.set_xlabel("Eb/N0 (dB)"); ax.set_ylabel("Bit Error Rate")
ax.set_title("Polar Coded BPSK BER Comparison")
ax.legend(fontsize=7); ax.grid(True, which="both", ls="--", alpha=0.5); ax.set_ylim(bottom=1e-5)
plt.show()


## 课后实践

请补全下方 Polar 编解码链路中的 **3 处空缺**（每处一行代码），完成编码→AWGN→LLR→解码的完整流程。

要求：

1. 补全 Polar 编码
2. 补全码率补偿的噪声方差计算
3. 补全 SC 解码

完成后运行 ，验证译码结果是否与原始信息一致。

In [ ]:
%%writefile polar_link_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.common.polar import PolarEncoder, get_polar_decoder

N, K = 256, 112
snr_db = 4.0
rng = np.random.default_rng(42)

enc = PolarEncoder(N, K)
dec = get_polar_decoder(N, K)
info = rng.integers(0, 2, size=K, dtype=np.int8)

# ==== TODO: 补全编码+信道+解码（3处空缺）====
# 步骤 1: Polar 编码
coded = ______________

# 步骤 2: BPSK 调制 + AWGN（含码率补偿）
bpsk = 1.0 - 2.0 * coded.astype(np.float64)
R_val = K / N
snr_lin = 10.0 ** (snr_db / 10.0)
noise_var = ______________     # 码率补偿的噪声方差
noise = rng.normal(0, np.sqrt(noise_var), size=N)
received = bpsk + noise
llr = 2.0 * received / noise_var

# 步骤 3: SC 解码
decoded = ______________

errors = int(np.sum(decoded != info))
print(f"SNR={snr_db:.0f} dB | N={N} K={K} | errors={errors}/{K}")
print("PASS" if errors == 0 else f"FAIL — {errors} bit errors")


执行以下命令进行编译并验证结果：


In [ ]:
!python polar_link_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/02.04_answer.txt
